# Weekly Hands-on Lab: NN Training (ReLU, Adam vs SGD)

This notebook is a working lab scaffold for the weekly assignment.

## Notebook references
- Baseline: https://github.com/samx18/nn-training-demo/blob/main/minimal_pytorch_nn_training.ipynb
- Session experiments: https://github.com/samx18/nn-training-demo/blob/main/minimal_pytorch_nn_training_variations.ipynb
- Real-life-like dataset (house prices): https://github.com/samx18/nn-training-demo/blob/main/minimal_pytorch_nn_training_variations_house_prices.ipynb

> Goal: reproduce good **ReLU + Adam** behavior and try to match it using **SGD** with tuning.


In [ ]:
# (Optional) Quick link checker for the three GitHub notebook URLs.
# Note: this needs internet access in your runtime.

import requests

urls = [
    "https://github.com/samx18/nn-training-demo/blob/main/minimal_pytorch_nn_training.ipynb",
    "https://github.com/samx18/nn-training-demo/blob/main/minimal_pytorch_nn_training_variations.ipynb",
    "https://github.com/samx18/nn-training-demo/blob/main/minimal_pytorch_nn_training_variations_house_prices.ipynb",
]

for u in urls:
    try:
        r = requests.get(u, timeout=15)
        print(r.status_code, u)
    except Exception as e:
        print("ERROR", u, "->", e)


## Experiment plan

Run and document at least these experiments:

| Experiment | Activation | Optimizer | LR | Momentum | Notes |
|---|---|---|---:|---:|---|
| E1 | ReLU | Adam | 0.001 | N/A | Baseline |
| E2 | ReLU | SGD | 0.1 | 0.0 | High LR check |
| E3 | ReLU | SGD | 0.01 | 0.9 | Common setup |
| E4 | ReLU | SGD | 0.001 | 0.9 | Lower LR stability |
| E5 | ReLU | SGD | tuned | tuned | Best SGD attempt |


In [ ]:
# Minimal PyTorch training scaffold (synthetic regression with noise)
# Use this if you want a local baseline independent of external notebooks.

import torch
import torch.nn as nn
import torch.optim as optim

# Reproducibility
torch.manual_seed(42)

# Noisy synthetic data
n = 1000
x = torch.rand(n, 1) * 10 - 5
y = 3 * x + 2 + torch.randn(n, 1) * 2.0

# Train/val split
split = int(0.8 * n)
x_train, x_val = x[:split], x[split:]
y_train, y_val = y[:split], y[split:]

class SimpleNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, 16),
            nn.ReLU(),
            nn.Linear(16, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
        )

    def forward(self, z):
        return self.net(z)

def run_experiment(optimizer_name='adam', lr=1e-3, momentum=0.0, epochs=200):
    model = SimpleNN()
    criterion = nn.MSELoss()

    if optimizer_name.lower() == 'adam':
        optimizer = optim.Adam(model.parameters(), lr=lr)
    elif optimizer_name.lower() == 'sgd':
        optimizer = optim.SGD(model.parameters(), lr=lr, momentum=momentum)
    else:
        raise ValueError('optimizer_name must be adam or sgd')

    history = []
    for epoch in range(epochs):
        model.train()
        pred = model(x_train)
        loss = criterion(pred, y_train)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            val_loss = criterion(model(x_val), y_val).item()

        history.append((loss.item(), val_loss))

    return model, history


In [ ]:
# Baseline run: ReLU + Adam
_, adam_hist = run_experiment(optimizer_name='adam', lr=1e-3, epochs=200)
print('Adam final train/val loss:', adam_hist[-1])

# Try SGD variant
_, sgd1_hist = run_experiment(optimizer_name='sgd', lr=1e-2, momentum=0.9, epochs=200)
print('SGD (lr=1e-2, m=0.9) final train/val loss:', sgd1_hist[-1])


## What to submit
For each run, record:
1. Hyperparameters (optimizer, LR, momentum, epochs, batch size if used).
2. Final train/validation loss.
3. Stability notes (oscillation/divergence/convergence speed).
4. Overfitting/underfitting observations.
5. Your best SGD setup and how close it gets to Adam.
